In [71]:
import polars as pl
from sklearn.model_selection import train_test_split
import os
from pathlib import Path

In [72]:
df = pl.read_csv("hf://datasets/dllllb/transactions-gender/transactions.csv.gz")
targets = pl.read_csv("hf://datasets/dllllb/transactions-gender/gender_train.csv")


In [73]:
df_time = (
    df.with_columns(
        pl.col("tr_datetime").str.split(" ").list.to_struct(fields=["day", "time"])
    )
    .unnest("tr_datetime")
    .with_columns(
        day=pl.duration(days=pl.col("day").cast(int)),
        time=pl.col("time").str.strptime(pl.Time, "%H:%M:%S") - pl.time(0, 0, 0),
    )
    .with_columns(tr_datetime=pl.col("day") + pl.col("time"))
    .drop("day", "time")
)

In [74]:
df_filt = df_time.filter(
    pl.len().over("customer_id").is_between(32, 1024),
    (pl.col("tr_datetime").max() - pl.col("tr_datetime").min()).over("customer_id")
    > 400,
).select(
    "customer_id",
    time=(pl.col("tr_datetime") - pl.col("tr_datetime").min()).over("customer_id")
    / (pl.col("tr_datetime").max() - pl.col("tr_datetime").min())
    .over("customer_id")
    .median(),
    mcc_code=pl.col("mcc_code").rank("dense").cast(pl.Int32),
    amount=pl.col("amount").abs().log1p() * pl.col("amount").sign(),
)

In [75]:
df_filt["mcc_code"].n_unique()

184

In [78]:
df_gb = df_filt.group_by("customer_id").agg(
    "time",
    "mcc_code",
    "amount",
)

df_target = df_gb.join(targets, on="customer_id").rename({"gender": "target"})

In [81]:
trainval, test = train_test_split(df_target, test_size=0.2, random_state=42)
train, val = train_test_split(trainval, test_size=0.2, random_state=42)
df_splits = pl.concat(
    [
        train.with_columns(split=pl.lit("train")),
        val.with_columns(split=pl.lit("val")),
        test.with_columns(split=pl.lit("test")),
    ]
)

df_splits.write_parquet(Path(os.environ["DATA_DIR"]) / "preprocessed/gender.parquet")